# Feature Engineering

---

In this step, we turn raw data into numerical features. This removes text "noise" and helps the model find patterns to predict difficulty.

In [1]:
# 1. Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style for the poster-ready visualizations
plt.style.use('fivethirtyeight')
sns.set_theme(style="whitegrid")

# 2. Load the prepared bestiary dataset 
# This uses the cleaned version we moved to the proposed folder
prepared_df = pd.read_csv('../../data/proposed/prepared_bestiary.csv')

# 3. Basic visualization and inspection
print("--- Dataset Information ---")
prepared_df.info()

print("\n--- Statistical Summary ---")
display(prepared_df.describe())

print("\n--- First 5 Records ---")
display(prepared_df.head())

--- Dataset Information ---
<class 'pandas.DataFrame'>
RangeIndex: 3658 entries, 0 to 3657
Data columns (total 36 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Name                    3658 non-null   str    
 1   Source                  3658 non-null   str    
 2   Page                    3658 non-null   str    
 3   Size                    3658 non-null   str    
 4   Type                    3658 non-null   str    
 5   Alignment               3658 non-null   str    
 6   AC                      3658 non-null   int64  
 7   HP                      3658 non-null   int64  
 8   Speed                   3658 non-null   str    
 9   Strength                3658 non-null   int64  
 10  Dexterity               3658 non-null   int64  
 11  Constitution            3658 non-null   int64  
 12  Intelligence            3658 non-null   int64  
 13  Wisdom                  3658 non-null   int64  
 14  Charisma               

,AC,HP,Strength,Dexterity,Constitution,Intelligence,Wisdom,Charisma,CR
count,3658.000000,3658.000000,3658.000000,3658.000000,3658.000000,3658.000000,3658.000000,3658.000000,3658.000000
mean,14.685894,96.116184,15.120831,13.423455,15.318480,10.694369,12.800164,11.664297,5.986844
std,2.965027,99.855364,6.004524,3.339387,4.186191,5.678043,3.550985,5.646058,6.189269
min,5.000000,1.000000,1.000000,1.000000,3.000000,1.000000,1.000000,1.000000,0.000000
25%,12.000000,27.000000,11.000000,11.000000,12.000000,6.000000,10.000000,7.000000,1.000000
50%,15.000000,66.000000,15.000000,14.000000,14.000000,11.000000,12.000000,11.000000,4.000000
75%,17.000000,135.000000,19.000000,15.000000,18.000000,14.000000,15.000000,16.000000,9.000000
max,25.000000,725.000000,30.000000,30.000000,30.000000,30.000000,30.000000,30.000000,30.000000



--- First 5 Records ---


,Name,Source,Page,Size,Type,Alignment,AC,HP,Speed,Strength,...,Bonus Actions,Reactions,Legendary Actions,Mythic Actions,Lair Actions,Regional Effects,Environment,Treasure,HP_Formula,Subtype
0,the demogorgon,imr,53,large,giant,chaotic neutral,15,123,40 ft.,21,...,none,none,none,none,none,none,unknown,none,13d12 + 39,none
1,aarakocra,mm14,12,medium,humanoid,neutral good,12,13,"20 ft., fly 50 ft.",10,...,none,none,none,none,none,none,mountain,none,3d8,aarakocra
2,aarakocra aeromancer,mm25,10,medium,elemental,neutral,16,66,"20 ft., fly 50 ft.",10,...,none,feather fall (1/day). the aarakocra casts feat...,none,none,none,none,"mountain, planar (elemental plane of air)","individual, implements",12d8 + 12,none
3,aarakocra simulacrum,skt,188,medium,humanoid,neutral good,12,6,"20 ft., fly 50 ft.",10,...,none,none,none,none,none,none,unknown,none,3d4,aarakocra
4,aarakocra skirmisher,mm25,10,medium,elemental,neutral,12,11,"20 ft., fly 50 ft.",10,...,none,none,none,none,none,none,"mountain, planar (elemental plane of air)","individual, implements",2d8 + 2,none


## Phase 1: Categorical & Ordinal Encoding
This stage turns descriptions into numbers or rankings.

### 1.1 Size (Ordinal Encoding)
* **Method:** Map categories to a **1–6 scale**: Tiny (1) to Gargantuan (6).
* **Justification:** Size has a natural order. Larger creatures usually have higher stats.

In [2]:
# Create the mapping dictionary
size_map = {
    'tiny': 1,
    'small': 2,
    'medium': 3,
    'large': 4,
    'huge': 5,
    'gargantuan': 6,
    'small or medium': 3, # Handling composite sizes found in the data
    'huge or gargantuan': 6
}

# Apply the mapping to the 'Size' column
# We use str.lower() to ensure it matches the keys accurately
prepared_df['Size_Rank'] = prepared_df['Size'].str.lower().map(size_map)

# Print the first 5 records to verify the transformation
print("--- Size Mapping Results ---")
print(prepared_df[['Name', 'Size', 'Size_Rank']].head())

# Show summary of the new ranking
print("\n--- Distribution of Size Rankings ---")
print(prepared_df['Size_Rank'].value_counts().sort_index())

--- Size Mapping Results ---
                   Name    Size  Size_Rank
0        the demogorgon   large          4
1             aarakocra  medium          3
2  aarakocra aeromancer  medium          3
3  aarakocra simulacrum  medium          3
4  aarakocra skirmisher  medium          3

--- Distribution of Size Rankings ---
Size_Rank
1     146
2     281
3    2013
4     688
5     365
6     165
Name: count, dtype: int64


### 1.2 Type (Top One-Hot Encoding)
* **Method:** Use the **80/20 Rule**. Create columns for the top types (Humanoid, Dragon, etc.).
* **Action:** Group all other types into an **"Other_Type"** bucket.
* **Justification:** This prevents the model from overfitting on rare types that only appear a few times.

In [3]:
# Calculate counts for each monster type
type_counts = prepared_df['Type'].value_counts().reset_index()
type_counts.columns = ['Monster Type', 'Count']

# Calculate Percentage and Cumulative Percentage
total_monsters = type_counts['Count'].sum()
type_counts['Percentage (%)'] = (type_counts['Count'] / total_monsters * 100).round(1)
type_counts['Cumulative %'] = type_counts['Percentage (%)'].cumsum().round(1)

# Highlight the Decision point (90% threshold)
type_counts['Decision'] = type_counts['Cumulative %'].apply(
    lambda x: 'Keep' if x <= 91.0 else 'Group into Other'
)

# Display the proof table
print("--- Type Distribution & Coverage Proof ---")
print(type_counts.head(15).to_string(index=False))

--- Type Distribution & Coverage Proof ---
        Monster Type  Count  Percentage (%)  Cumulative %         Decision
            humanoid   1206            33.0          33.0             Keep
         monstrosity    369            10.1          43.1             Keep
              undead    304             8.3          51.4             Keep
           construct    264             7.2          58.6             Keep
               beast    252             6.9          65.5             Keep
               fiend    234             6.4          71.9             Keep
          aberration    194             5.3          77.2             Keep
              dragon    191             5.2          82.4             Keep
               giant    161             4.4          86.8             Keep
           elemental    117             3.2          90.0             Keep
                 fey    115             3.1          93.1 Group into Other
           celestial     91             2.5          95.6

In [4]:
# Define our evidence-based top types
top_types = [
    'humanoid', 'monstrosity', 'undead', 'construct', 
    'beast', 'fiend', 'aberration', 'dragon', 
    'giant', 'elemental'
]

# Create a grouped column where non-top types are labeled 'other'
prepared_df['Grouped_Type'] = prepared_df['Type'].str.lower().apply(
    lambda x: x if x in top_types else 'other'
)

# Generate one-hot encoded columns with the 'Type' prefix
type_dummies = pd.get_dummies(prepared_df['Grouped_Type'], prefix='Type').astype(int)

# Combine with the main dataframe
prepared_df = pd.concat([prepared_df, type_dummies], axis=1)

# Verify the uniform naming
print(type_dummies.columns.tolist())

['Type_aberration', 'Type_beast', 'Type_construct', 'Type_dragon', 'Type_elemental', 'Type_fiend', 'Type_giant', 'Type_humanoid', 'Type_monstrosity', 'Type_other', 'Type_undead']


In [5]:
# 1. Check data types of the new columns
print("--- New Column Data Types ---")
type_cols = [col for col in prepared_df.columns if col.startswith('Type_')]
print(prepared_df[type_cols].dtypes)

# 2. View a sample to compare the original 'Type' with the encoded columns
# We include 'Grouped_Type' to see how the 'other' logic worked
print("\n--- Encoding Sample (Original vs. Encoded) ---")
verification_cols = ['Type', 'Grouped_Type'] + type_cols
print(prepared_df[verification_cols].head(10))

--- New Column Data Types ---
Type_aberration     int64
Type_beast          int64
Type_construct      int64
Type_dragon         int64
Type_elemental      int64
Type_fiend          int64
Type_giant          int64
Type_humanoid       int64
Type_monstrosity    int64
Type_other          int64
Type_undead         int64
dtype: object

--- Encoding Sample (Original vs. Encoded) ---
        Type Grouped_Type  Type_aberration  Type_beast  Type_construct  \
0      giant        giant                0           0               0   
1   humanoid     humanoid                0           0               0   
2  elemental    elemental                0           0               0   
3   humanoid     humanoid                0           0               0   
4  elemental    elemental                0           0               0   
5   humanoid     humanoid                0           0               0   
6      plant        other                0           0               0   
7      plant        other     

### 1.3 Swarm Consolidation
* **Method:** Create a binary flag `is_swarm` and map swarms back to their "Parent" type.
* **Action:** Search for the keyword "swarm" to set a 1 or 0 flag. Then, extract the base creature type (e.g., "Beast") so the monster is counted in the correct category.
* **Justification:** Swarms share the same fundamental stats as their parent type but have higher HP and unique resistances. This flag allows the model to learn that "Swarm-ness" is a power modifier.

In [6]:
# 1. Create the binary is_swarm flag
# This captures the tactical 'Swarm' mechanic regardless of the creature type
prepared_df['is_swarm'] = prepared_df['Type'].str.contains('swarm', case=False, na=False).astype(int)

# 2. Extract the "Base Type" to fix the One-Hot encoding
# We want a "Swarm of Tiny Beasts" to count as a 'beast'
def get_base_type(type_str):
    type_str = str(type_str).lower()
    if 'swarm' in type_str:
        # Check if any of our top_types are mentioned in the swarm description
        for t in top_types:
            if t in type_str:
                return t
        return 'other'
    return type_str if type_str in top_types else 'other'

# Update the Grouped_Type using our new base-type logic
prepared_df['Grouped_Type'] = prepared_df['Type'].apply(get_base_type)

# 3. Re-run One-Hot Encoding with the cleaned categories
# We use 'Type_other' for the bucket to keep it uniform
type_dummies = pd.get_dummies(prepared_df['Grouped_Type'], prefix='Type').astype(int)

# 4. Clean up: Remove any old Type_ columns and add the new corrected ones
prepared_df = prepared_df.drop(columns=[c for c in prepared_df.columns if c.startswith('Type_')])
prepared_df = pd.concat([prepared_df, type_dummies], axis=1)

# Verification
print("--- Swarm Consolidation Results ---")
verification = prepared_df[prepared_df['is_swarm'] == 1][['Type', 'Grouped_Type', 'is_swarm']].head()
print(verification)

print("\nFinal Uniform Type Columns:")
print([col for col in prepared_df.columns if col.startswith('Type_')])

--- Swarm Consolidation Results ---
                           Type Grouped_Type  is_swarm
296        swarm of tiny fiends        fiend         1
674   swarm of tiny aberrations   aberration         1
2235  swarm of tiny aberrations   aberration         1
2391   swarm of tiny constructs    construct         1
2682       swarm of tiny beasts        beast         1

Final Uniform Type Columns:
['Type_aberration', 'Type_beast', 'Type_construct', 'Type_dragon', 'Type_elemental', 'Type_fiend', 'Type_giant', 'Type_humanoid', 'Type_monstrosity', 'Type_other', 'Type_undead']


## Phase 2: Feature Extraction (Numerical & Boolean)
We extract specific mechanical values from text columns.

### 2.1 Complexity Scores
* **Trait_Count & Action_Count:** Count the items in these columns.
* **Justification:** High-tier monsters have more options. This measures "Action Economy."

### Trait Count Extraction
* **Method:** We split the text in the `Traits` column by the newline character (`\n`) and count the resulting items.
* **Action:** If the column contains the word "none" or is empty, we return a count of **0**.
* **Justification:** Each distinct trait in the dataset is stored on a new line. Counting these lines gives us a direct measure of a monster's complexity and "Action Economy."

In [7]:
# Function to count traits
def get_trait_count(trait_text):
    """
    Counts the number of distinct traits listed for a creature.
    Assumes each trait is separated by a newline character.
    """
    if pd.isna(trait_text) or str(trait_text).lower() == 'none':
        return 0
    
    # Split by newline and remove any empty strings from the list
    traits = [t for t in str(trait_text).split('\n') if t.strip()]
    return len(traits)

# Apply the trait count logic to the dataframe
prepared_df['Trait_Count'] = prepared_df['Traits'].apply(get_trait_count)

# Select a sample of rows with and without traits
trait_samples = prepared_df[['Name', 'Trait_Count', 'Traits']].head(10)

print("--- Verification Table: Trait Count Extraction ---")
print(trait_samples.to_string(index=False, max_colwidth=60))

--- Verification Table: Trait Count Extraction ---
                 Name  Trait_Count                                                       Traits
       the demogorgon            2 two heads. the ettin has advantage on wisdom (perception)...
            aarakocra            1 dive attack. if the aarakocra is flying and dives at leas...
 aarakocra aeromancer            0                                                         none
 aarakocra simulacrum            2 dive attack. if the aarakocra is flying and dives at leas...
 aarakocra skirmisher            0                                                         none
aarakocra spelljammer            1 spellcasting. the aarakocra is a 9th-level spellcaster. i...
         aartuk elder            1 spider climb. the aartuk can climb difficult surfaces, in...
    aartuk starhorror            1 spider climb. the aartuk can climb difficult surfaces, in...
      aartuk weedling            1 spider climb. the aartuk can climb difficult surfa

### let's do action count next
In D&D 5e, "Action Economy" is the single biggest predictor of threat level. However, this data is buried in unstructured text. We use Regular Expressions (Regex) to solve four specific challenges:

* **Multi-sentence logic:** Identifying the "Multiattack" section specifically to avoid counting unrelated flavor text.
* **Digit-Word Mapping:** Converting written numbers ("three") and digits ("3") into a uniform integer.
* **Additive Math:** Summing individual components (e.g., "one bite and two claws") when a total isn't explicitly stated.
* **Implicit 1:** Defaulting to a count of 1 for monsters that lack a Multiattack trait but still have standard actions.

In [8]:
import re

def get_attack_count(action_text):
    """
    Extracts the number of attacks a monster makes during a Multiattack action.
    Defaults to 1 if no Multiattack is found.
    """
    # 1. Handle missing data or simple single-action monsters
    if pd.isna(action_text) or 'multiattack' not in str(action_text).lower():
        return 1
    
    # 2. Isolate the 'Multiattack' sentence to prevent 'noise' from other actions
    multi_part = re.search(r'multiattack\.(.*?)(?:\n|\. [A-Z]|$)', str(action_text), re.I | re.S)
    if not multi_part: 
        return 1
    
    sentence = multi_part.group(1).lower()

    # Dictionary to normalize natural language to integers
    word_to_num = {
        'one': 1, 'two': 2, 'three': 3, 'four': 4, 'five': 5, 'six': 6,
        '1': 1, '2': 2, '3': 3, '4': 4, '5': 5, '6': 6
    }

    # Pattern A: Direct total (e.g., "makes three attacks" or "makes 2 attacks")
    match_total = re.search(r'(?:makes|takes)\s+(one|two|three|four|five|six|\d+)\s+(?:[\w\s]*?)\s*attacks?', sentence)
    if match_total:
        return word_to_num.get(match_total.group(1), 1)

    # Pattern B: Additive sum (e.g., "one bite and two claw attacks")
    # We find all counts associated with the word 'attack' and sum them
    counts = re.findall(r'(one|two|three|four|five|six|\d+)\s+(?:[\w\s]*?)\s*attacks?', sentence)
    if counts:
        return sum(word_to_num.get(c, 0) for c in counts)

    # Default fallback for complex Multiattacks (e.g., "uses its breath weapon and moves")
    return 1

# Apply the logic to the dataframe
prepared_df['Action_Count'] = prepared_df['Actions'].apply(get_attack_count)

# Verification: Look at the top 10 to see the extraction in action
print("--- Action Count Extraction Samples ---")
print(prepared_df[['Name', 'Action_Count', 'Actions']].head(10))

--- Action Count Extraction Samples ---
                    Name  Action_Count  \
0         the demogorgon             2   
1              aarakocra             1   
2   aarakocra aeromancer             2   
3   aarakocra simulacrum             1   
4   aarakocra skirmisher             1   
5  aarakocra spelljammer             1   
6           aartuk elder             2   
7      aartuk starhorror             2   
8        aartuk weedling             2   
9       aberrant cultist             2   

                                             Actions  
0  multiattack. the ettin makes two attacks: one ...  
1  talon. melee weapon attack: +4 to hit, reach 5...  
2  multiattack. the aarakocra makes two wind staf...  
3  talon. melee weapon attack: +4 to hit, reach 5...  
4  talons. melee attack roll: +4, reach 5 ft. hit...  
5  dagger. melee or ranged weapon attack: +5 to h...  
6  multiattack. the aartuk makes two branch attac...  
7  multiattack. the aartuk makes two branch attac...  
8 

### 2.2 Binary Flags
* **Is_Legendary:** 1 if they have legendary actions, else 0.
* **Is_Spellcaster:** 1 if "spellcasting" is in the text, else 0.

In [9]:
# 1. Extract Is_Legendary flag
# We check if the 'Legendary Actions' column has any content other than 'none'
prepared_df['Is_Legendary'] = prepared_df['Legendary Actions'].apply(
    lambda x: 0 if pd.isna(x) or str(x).lower() == 'none' else 1
)

# 2. Extract Is_Spellcaster flag (Bonus Extraction)
# Spellcasting is another major power multiplier. 
# We look for the "Spellcasting" trait in the Traits or Actions columns.
prepared_df['Is_Spellcaster'] = prepared_df.apply(
    lambda row: 1 if 'spellcasting' in str(row['Traits']).lower() or 
                     'spellcasting' in str(row['Actions']).lower() else 0, 
    axis=1
)

# Verification
print("--- Boss Flag Extraction Samples ---")
print(prepared_df[['Name', 'CR', 'Is_Legendary', 'Is_Spellcaster']].sort_values(by='CR', ascending=False).head(10))

--- Boss Flag Extraction Samples ---
                                          Name    CR  Is_Legendary  \
3089                                 tarrasque  30.0             1   
3166                                    tiamat  30.0             1   
1027                                   emrakul  30.0             1   
242                           aspect of tiamat  30.0             1   
241                          aspect of bahamut  30.0             1   
991   elder dinosaur (zacama, primal calamity)  30.0             1   
992      elder dinosaur (zetalpa, primal dawn)  30.0             1   
240                                   asmodeus  30.0             1   
986                             elder dinosaur  30.0             1   
987       elder dinosaur (etali, primal storm)  30.0             1   

      Is_Spellcaster  
3089               0  
3166               1  
1027               0  
242                0  
241                0  
991                0  
992                0  
240     

### 2.3 Movement
* **Action:** Create a **Max_Speed** feature from all movement types (Fly, Swim, etc.).
* **Justification:** Flying or burrowing are major power multipliers in combat.

In [10]:
# Check unique values and their frequency for the HP column
speed_counts = prepared_df['Speed'].value_counts()

# Print the top 20 most common HP formats
print(f"Total Unique Speed entries: {len(speed_counts)}")
print("-" * 30)
print(speed_counts.head(50))

Total Unique Speed entries: 295
------------------------------
Speed
30 ft.                                             1434
40 ft.                                              347
30 ft., swim 30 ft.                                 108
25 ft.                                              107
30 ft., climb 30 ft.                                104
20 ft.                                              103
50 ft.                                               71
30 ft., fly 60 ft.                                   50
40 ft., climb 40 ft.                                 47
20 ft., climb 20 ft.                                 36
30 ft., fly 30 ft. (hover)                           36
60 ft.                                               36
30 ft., fly 30 ft.                                   36
10 ft., fly 60 ft.                                   33
40 ft., fly 60 ft.                                   28
40 ft., fly 80 ft., swim 40 ft.                      27
40 ft., climb 40 ft., fly 80 ft.   

In [11]:
# --- Phase 2.8: Speed Feature Extraction ---
import re
import pandas as pd

def extract_speed_data(speed_str):
    # Handle NaN or 'none' strings safely
    if pd.isna(speed_str) or str(speed_str).strip().lower() == "none":
        return 0, "None"
    
    # Ensure we are working with a string
    speed_str = str(speed_str)
    
    # 1. Extract all distances (e.g., '30', '60') and find the maximum
    distances = [int(n) for n in re.findall(r'(\d+)\s*ft', speed_str)]
    max_dist = max(distances) if distances else 0
    
    # 2. Identify movement categories
    modes = []
    # If the string starts with a number (e.g., "30 ft."), it's the standard 'walk' speed
    if re.match(r'^\d+', speed_str):
        modes.append('walk')
    
    # Check for specific movement keywords
    for mode in ['fly', 'climb', 'swim', 'burrow']:
        if re.search(mode, speed_str, re.IGNORECASE):
            modes.append(mode)
            
    # Return 0 and "None" if no modes were found, otherwise join them
    return max_dist, (", ".join(modes) if modes else "None")

# Update this line to use prepared_df instead of df
prepared_df[['Max_Speed', 'Movement_Types']] = prepared_df['Speed'].apply(lambda x: pd.Series(extract_speed_data(x)))

# Convert Max_Speed to integer
prepared_df['Max_Speed'] = prepared_df['Max_Speed'].astype(int)

# --- Concise Verification ---
print(f"Max_Speed Dtype: {prepared_df['Max_Speed'].dtype}")
print(f"Deep Check:      {prepared_df['Max_Speed'].apply(type).unique()} (Should be <class 'int'>)")
print("-" * 30)
print("Sample Results:")
print(prepared_df[['Speed', 'Max_Speed', 'Movement_Types']].head(10))

Max_Speed Dtype: int64
Deep Check:      [<class 'int'>] (Should be <class 'int'>)
------------------------------
Sample Results:
                  Speed  Max_Speed Movement_Types
0                40 ft.         40           walk
1    20 ft., fly 50 ft.         50      walk, fly
2    20 ft., fly 50 ft.         50      walk, fly
3    20 ft., fly 50 ft.         50      walk, fly
4    20 ft., fly 50 ft.         50      walk, fly
5    30 ft., fly 50 ft.         50      walk, fly
6  20 ft., climb 20 ft.         20    walk, climb
7  20 ft., climb 20 ft.         20    walk, climb
8  20 ft., climb 20 ft.         20    walk, climb
9                30 ft.         30           walk


In [ ]:
import re

def extract_speed_data(speed_str):
    """
    Parses the Speed string to extract Max_Speed, list the modes, and count them.
    """
    if pd.isna(speed_str) or str(speed_str).strip().lower() == "none":
        return 0, "None", 0
    
    speed_str = str(speed_str)
    
    # 1. Extract all distances (e.g., '30', '60') and find the maximum
    distances = [int(n) for n in re.findall(r'(\d+)\s*ft', speed_str)]
    max_dist = max(distances) if distances else 0
    
    # 2. Identify movement categories
    modes = []
    # If the string starts with a number, it's the standard 'walk' speed
    if re.match(r'^\d+', speed_str):
        modes.append('walk')
    
    # Check for specific movement keywords
    for mode in ['fly', 'climb', 'swim', 'burrow']:
        if re.search(mode, speed_str, re.IGNORECASE):
            modes.append(mode)
            
    # Return: Max Distance, Mode String, and the Count of modes
    return max_dist, (", ".join(modes) if modes else "None"), len(modes)

# Apply the function and create three new columns at once
prepared_df[['Max_Speed', 'Movement_Types', 'Movement_Count']] = prepared_df['Speed'].apply(
    lambda x: pd.Series(extract_speed_data(x))
)

# Ensure numerical columns are the correct type
prepared_df['Max_Speed'] = prepared_df['Max_Speed'].astype(int)
prepared_df['Movement_Count'] = prepared_df['Movement_Count'].astype(int)

# --- Verification ---
print("--- Movement Feature Verification ---")
print(prepared_df[['Name', 'Max_Speed', 'Movement_Types', 'Movement_Count']].head(10))

--- Movement Feature Verification ---
                    Name                 Speed  Max_Speed Movement_Types  \
0         the demogorgon                40 ft.         40           walk   
1              aarakocra    20 ft., fly 50 ft.         50      walk, fly   
2   aarakocra aeromancer    20 ft., fly 50 ft.         50      walk, fly   
3   aarakocra simulacrum    20 ft., fly 50 ft.         50      walk, fly   
4   aarakocra skirmisher    20 ft., fly 50 ft.         50      walk, fly   
5  aarakocra spelljammer    30 ft., fly 50 ft.         50      walk, fly   
6           aartuk elder  20 ft., climb 20 ft.         20    walk, climb   
7      aartuk starhorror  20 ft., climb 20 ft.         20    walk, climb   
8        aartuk weedling  20 ft., climb 20 ft.         20    walk, climb   
9       aberrant cultist                30 ft.         30           walk   

   Movement_Count  
0               1  
1               2  
2               2  
3               2  
4               2  
5    

### 2.3 Avg_HP & Max_HP
* **Method:**
    * **Average HP:** Taken directly from the `HP` column and converted to an integer.
    * **Maximum HP:** Calculated from the `HP_Formula` column ($X\text{d}Y + Z$) using the formula: $(X \times Y) + Z$.
* **Action:**
    * Use **Regex** to isolate the number of dice ($X$), the die size ($Y$), and the optional modifier ($Z$).
    * If the formula is marked as "static" or is missing, the Maximum HP defaults to the Average HP.
* **Justification:** While Average HP gives a baseline, the Max HP represents the upper bound of a monster's durability. The difference between these two values provides a measure of **"HP Variance,"** which is often higher for elite monsters.

In [12]:
import re

def calculate_max_hp(row):
    """
    Parses a D&D HP formula (e.g., '13d12 + 39' or '3d8') 
    and returns the maximum possible value.
    If the formula is 'static', it uses the provided average HP.
    """
    hp_formula = str(row['HP_Formula']).lower().strip()
    avg_hp = int(row['HP'])
    
    # Handle cases without a dice formula
    if pd.isna(hp_formula) or hp_formula in ['none', '', 'static']:
        return avg_hp
    
    # Regex to find: (NumDice)d(DieSides) + (Bonus)
    # Example: '13d12 + 39' -> Group 1: 13, Group 2: 12, Group 3: 39
    match = re.search(r'(\d+)d(\d+)(?:\s*\+\s*(\d+))?', hp_formula)
    
    if match:
        num_dice = int(match.group(1))
        die_sides = int(match.group(2))
        bonus = int(match.group(3)) if match.group(3) else 0
        return (num_dice * die_sides) + bonus
    
    # Fallback for simple numeric-only formulas
    try:
        return int(hp_formula)
    except:
        return avg_hp

# 1. Create the Average HP column
prepared_df['Avg_HP'] = prepared_df['HP'].astype(int)

# 2. Apply the Max HP calculation logic
prepared_df['Max_HP'] = prepared_df.apply(calculate_max_hp, axis=1)

# Verification
print("--- HP Feature Extraction Samples ---")
print(prepared_df[['Name', 'HP_Formula', 'Avg_HP', 'Max_HP']].head(5))

--- HP Feature Extraction Samples ---
                   Name  HP_Formula  Avg_HP  Max_HP
0        the demogorgon  13d12 + 39     123     195
1             aarakocra         3d8      13      24
2  aarakocra aeromancer   12d8 + 12      66     108
3  aarakocra simulacrum         3d4       6      12
4  aarakocra skirmisher     2d8 + 2      11      18


## Before we continue, let's first see all the columns we have now and what values and datatypes do they have

In [ ]:
# Updated Audit Cell to include Movement_Types
# 1. Summary of Data Types and Memory Usage
print("--- Dataset Information ---")
prepared_df.info()

# 2. Check for any remaining Nulls in our new features
new_cols = [
    'Size_Rank', 'is_swarm', 'Trait_Count', 'Action_Count', 
    'Is_Legendary', 'Is_Spellcaster', 'Max_Speed', 'Movement_Types', 'Avg_HP', 'Max_HP'
]

print("\n--- Null Value Check in New Features ---")
print(prepared_df[new_cols].isnull().sum())

# 3. View Sample Values
print("\n--- Sample Values for Engineered Features ---")
display(prepared_df[['Name', 'CR'] + new_cols].head(10))

# 4. Detailed Data Type Check
print("\n--- Precise Data Types ---")
print(prepared_df[new_cols].dtypes)

--- Dataset Information ---
<class 'pandas.DataFrame'>
RangeIndex: 3658 entries, 0 to 3657
Data columns (total 58 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Name                    3658 non-null   str    
 1   Source                  3658 non-null   str    
 2   Page                    3658 non-null   str    
 3   Size                    3658 non-null   str    
 4   Type                    3658 non-null   str    
 5   Alignment               3658 non-null   str    
 6   AC                      3658 non-null   int64  
 7   HP                      3658 non-null   int64  
 8   Speed                   3658 non-null   str    
 9   Strength                3658 non-null   int64  
 10  Dexterity               3658 non-null   int64  
 11  Constitution            3658 non-null   int64  
 12  Intelligence            3658 non-null   int64  
 13  Wisdom                  3658 non-null   int64  
 14  Charisma               

,Name,CR,Size_Rank,is_swarm,Trait_Count,Action_Count,Is_Legendary,Is_Spellcaster,Max_Speed,Movement_Types,Avg_HP,Max_HP
0,the demogorgon,8.000,4,0,2,2,0,0,40,walk,123,195
1,aarakocra,0.250,3,0,1,1,0,0,50,"walk, fly",13,24
2,aarakocra aeromancer,4.000,3,0,0,2,0,1,50,"walk, fly",66,108
3,aarakocra simulacrum,0.125,3,0,2,1,0,0,50,"walk, fly",6,12
4,aarakocra skirmisher,0.250,3,0,0,1,0,0,50,"walk, fly",11,18
5,aarakocra spelljammer,6.000,3,0,1,1,0,1,50,"walk, fly",40,72
6,aartuk elder,3.000,4,0,1,2,0,1,20,"walk, climb",75,120
7,aartuk starhorror,2.000,3,0,1,2,0,1,20,"walk, climb",52,80
8,aartuk weedling,2.000,3,0,1,2,0,0,20,"walk, climb",38,63
9,aberrant cultist,8.000,3,0,0,2,0,1,30,walk,137,225



--- Precise Data Types ---
Size_Rank         int64
is_swarm          int64
Trait_Count       int64
Action_Count      int64
Is_Legendary      int64
Is_Spellcaster    int64
Max_Speed         int64
Movement_Types      str
Avg_HP            int64
Max_HP            int64
dtype: object


## Phase 3: Feature Scaling & Standardization
This puts all numbers on the same mathematical scale.

### 3.1 Z-Score Standardization
* **Applied to:** AC, Ability Scores, and Max_Speed.
* **Formula:** $$z = \frac{x - \mu}{\sigma}$$
* **Justification:** These follow a normal distribution. This keeps them weighted equally.

### 3.2 Min-Max Scaling
* **Applied to:** HP and Trait_Count.
* **Justification:** HP has extreme outliers. Scaling to a **0–1 range** stops outliers from distorting the model weights.

----------------------------------------------------------------------

## Phase 4: Evidence-Based Pruning
We remove data that does not help the model learn.

### 4.1 Metadata Exclusion
* **Action:** Drop *Name, Source, Page, and HP_Formula*.
* **Justification:** These are identifiers, not mechanics.

### 4.2 Variance & Correlation
* **Low Variance:** Drop features where 99% of the values are the same.
* **High Correlation:** If two features (like Strength and Athletics) are $> 0.90$ correlated, remove one.
* **Justification:** This reduces redundancy and speeds up the model.